# SimulacraBench tutorial

Two parts, a few minutes, no GPU, nothing downloaded.

1. **A sandbox you can read.** `data/sample.json` is a toy instrument -- seven
   items, four hundred invented respondents -- small enough to print. It shows
   the shape of the task: what a schema declares, what a held-out respondent
   looks like, and what order your answers go back in.
2. **The baseline on a real instrument.** Score the shipped submission on
   UNICEF through `score.py`, exactly as the grader will.

Everything here runs on **invented practice data**. No real survey answer
appears in this notebook or anywhere else in the repository.

See `README.md` for the submission contract, the scoring rule and the rules.


In [1]:
import subprocess, sys

import numpy as np
import pandas as pd

# The same modules score.py uses. Nothing here reimplements the grader: when
# you score something in this notebook, you are calling the code that scores
# you.
from make_sandbox import (generated_items, load_config, load_schema,
                          options_for, scored_items, write_sandbox)
from score import floored, load_frames, sample_rows
from score import score as grade

SEED = 0
PHASE = 1
config = load_config("config.yml")

pd.set_option("display.width", 200, "display.max_columns", 50)


## 1. A sandbox you can read

`make_sandbox.py` turns a schema into a dataset of exactly the shape the real
one has. It writes three files: `train.parquet` (respondents whose answers
everybody sees), `test.parquet` (the ones being predicted), and `schema.json`
(what your `predict()` receives).

The split is decided here, once, and written to disk. `score.py` samples rows
from these files; it never splits anything itself.


In [2]:
sample = load_schema("data/sample.json", config)
write_sandbox(sample, config, "_sandbox/sample", seed=SEED)

train, test = load_frames("_sandbox/sample", sample)
print("train", train.shape, " test", test.shape)
print()
print(sample["dataset"]["description"])


train (302, 8)  test (98, 8)

A toy instrument, not a real survey. Seven items and four hundred invented respondents, small enough to print a sandbox and read it. It has one of everything the real schemas have: a frame block that is always visible, items that are scored, a gate chain two deep, and an EXCLUDE column the grader never shows anybody. Use it to see the shape of the task; use the three real schemas to see whether a method works.


### What the schema says

Four keys per item and no more. `class` decides what happens to it, `values`
lists the allowed answers, and `gate` says which earlier answer has to hold for
the question to be asked at all.


In [3]:
rows = []
for name, rec in sample["items"].items():
    gate = rec.get("gate") or {}
    rows.append({"item": name,
                 "class": rec["class"],
                 "K": len(options_for(sample, name)) if rec["values"] else 0,
                 "gate": gate.get("parent", "-"),
                 "asked if": ", ".join(gate.get("observed_if", [])) or "-",
                 "options": " | ".join(map(str, rec["values"] or ["-"]))})
print(pd.DataFrame(rows).to_string(index=False))


                  item   class  K           gate                                              asked if                                                 options
                region   GIVEN  3              -                                                     -                                 North | Central | South
              age_band   GIVEN  4              -                                                     -                             18-29 | 30-44 | 45-59 | 60+
household_has_children   GIVEN  2              -                                                     -                                                Yes | No
     interviewer_notes EXCLUDE  0              -                                                     -                                                       -
        visited_clinic PREDICT  3              -                                                     -                         Yes | No | Prefer not to answer
           clinic_wait PREDICT  4 visited_clin

`interviewer_notes` is `EXCLUDE`, so it is never generated, never shown and
never scored -- it does not appear in the data at all. `region`, `age_band` and
`household_has_children` are `GIVEN`: always visible, never scored. The other
four are `PREDICT`.

Note `K` on the two gated items. Their option list is one longer than `values`,
because **`NA_GATED` is an answer**. Someone who never visited a clinic was
never asked how long they waited, and predicting that is worth exactly as much
as predicting anything else.

### The data itself


In [4]:
train.head(10)


,respondent_id,region,age_band,household_has_children,visited_clinic,clinic_wait,would_return,trusts_health_advice
0,R000001,South,30-44,No,Prefer not to answer,NA_GATED,NA_GATED,Somewhat
1,R000003,South,60+,No,Prefer not to answer,NA_GATED,NA_GATED,A little
2,R000004,South,60+,Yes,Prefer not to answer,NA_GATED,NA_GATED,Somewhat
3,R000007,South,60+,Yes,Prefer not to answer,NA_GATED,NA_GATED,Somewhat
4,R000008,Central,18-29,Yes,Prefer not to answer,NA_GATED,NA_GATED,Somewhat
5,R000009,Central,30-44,No,Prefer not to answer,NA_GATED,NA_GATED,Somewhat
6,R000010,Central,30-44,No,No,NA_GATED,NA_GATED,Somewhat
7,R000011,Central,30-44,Yes,No,NA_GATED,NA_GATED,A little
8,R000014,South,18-29,Yes,Prefer not to answer,NA_GATED,NA_GATED,Somewhat
9,R000015,Central,18-29,No,Prefer not to answer,NA_GATED,NA_GATED,Somewhat


`would_return` gates on `clinic_wait`, which gates on `visited_clinic`: a chain
two deep. Anyone who answered `No` to the first carries `NA_GATED` in both of
the others, and the generator closes the chain without any special handling.


In [5]:
chain = ["visited_clinic", "clinic_wait", "would_return"]
print(train[chain].value_counts().to_frame("respondents").to_string())


                                                         respondents
visited_clinic       clinic_wait           would_return             
Prefer not to answer NA_GATED              NA_GATED              206
No                   NA_GATED              NA_GATED               43
Yes                  30 minutes to 2 hours No                     26
                                           Not sure               20
                                           Yes                     5
                     Under 30 minutes      No                      1
                                           Not sure                1


### What `predict()` is handed

`score.py` samples this phase's rows, stacks training respondents on top of
test respondents, and blanks every `PREDICT` cell of the latter. Those blanks
are what you return probabilities for.


In [6]:
frame_s, cells_s, truth_s = sample_rows(sample, train, test, config, PHASE,
                                       seed=SEED)
shown = [n for n, r in sample["items"].items() if r["class"] != "EXCLUDE"]
print("frame:", frame_s.shape, " cells to predict:", len(cells_s))
print()
print(pd.concat([frame_s[["respondent_id"] + shown].head(4),
                 frame_s[["respondent_id"] + shown].tail(4)]).to_string(index=False))


frame: (121, 8)  cells to predict: 144

respondent_id  region age_band household_has_children       visited_clinic clinic_wait would_return trusts_health_advice
      R000003   South      60+                     No Prefer not to answer    NA_GATED     NA_GATED             A little
      R000004   South      60+                    Yes Prefer not to answer    NA_GATED     NA_GATED             Somewhat
      R000008 Central    18-29                    Yes Prefer not to answer    NA_GATED     NA_GATED             Somewhat
      R000009 Central    30-44                     No Prefer not to answer    NA_GATED     NA_GATED             Somewhat
      R000358 Central      60+                     No                  NaN         NaN          NaN                  NaN
      R000362   South    18-29                    Yes                  NaN         NaN          NaN                  NaN
      R000373   South    18-29                     No                  NaN         NaN          NaN              

The top rows are training respondents: complete, and yours to learn from. The
bottom rows are held out -- you see `region`, `age_band` and
`household_has_children`, and nothing else. **`NaN` means "predict this" and
never means "they did not answer";** genuine non-response is an ordinary
option, like `Prefer not to answer` on `visited_clinic`.

`cells` is the canonical order your return value has to follow: rows top to
bottom, and within a row, items in schema key order.


In [7]:
print(pd.DataFrame(cells_s, columns=["row", "respondent_id", "item"]).head(9)
      .to_string(index=False))


 row respondent_id                 item
  85       R000006       visited_clinic
  85       R000006          clinic_wait
  85       R000006         would_return
  85       R000006 trusts_health_advice
  86       R000032       visited_clinic
  86       R000032          clinic_wait
  86       R000032         would_return
  86       R000032 trusts_health_advice
  87       R000035       visited_clinic


### Score something on it

The crowd baseline: every held-out respondent gets each item's smoothed shares.
`skill` is 0 for a uniform guess and 1 for perfection, and it is what the
leaderboard ranks. Higher is better.


In [8]:
def hidden_cells(frame, items):
    """Every blank cell, in the order predict() must return them."""
    values = frame[items].to_numpy(dtype=object)
    ids = frame["respondent_id"].to_numpy(dtype=object)
    return [(row, ids[row], items[col])
            for row in range(values.shape[0])
            for col in range(len(items))
            if pd.isna(values[row, col])]

def crowd_for(sch, frame):
    items = generated_items(sch)
    tables = {}
    for item in items:
        counts = frame[item].value_counts()
        n = np.array([counts.get(o, 0) for o in options_for(sch, item)], float)
        tables[item] = (n + 0.5) / (n + 0.5).sum()
    return [tables[item] for _, _, item in hidden_cells(frame, items)]

vec_s = floored(crowd_for(sample, frame_s), sample, cells_s, config["scoring"]["floor"])
res_s = grade(sample, config, vec_s, truth_s, cells_s)
print("uniform reference {:.4f} nats".format(res_s["uniform_reference"]))
print("log score         {:.4f}".format(res_s["log_score"]))
print("skill             {:.4f}   <- 0 is a uniform guess, 1 is perfect".format(res_s["skill"]))


uniform reference 1.3144 nats
log score         -0.6954
skill             0.4709   <- 0 is a uniform guess, 1 is perfect


## 2. Run the baseline on a real instrument

`baseline/marginal_counts` is a working submission: for each item it predicts
the smoothed shares of the answers it can see, and ignores everything about the
individual respondent. It is what you get if you copy nothing and change
nothing, and it is what you have to beat.

Build the UNICEF sandbox and score it through `score.py` -- fresh virtual
environment, dependencies installed, network cut, exactly as the grader does.


In [9]:
SCHEMA = "data/unicef.json"
schema = load_schema(SCHEMA, config)
write_sandbox(schema, config, "_sandbox/unicef", seed=SEED)

train_u, test_u = load_frames("_sandbox/unicef", schema)
print("%d items, %d scored" % (len(generated_items(schema)),
                               len(scored_items(schema))))
print("train %s   test %s" % (train_u.shape, test_u.shape))


19 items, 12 scored
train (14863, 20)   test (4984, 20)


In [10]:
done = subprocess.run(
    [sys.executable, "score.py",
     "--submission", "baseline/marginal_counts",
     "--data", "_sandbox/unicef",
     "--schema", SCHEMA,
     "--phase", "1"],
    capture_output=True, text=True)
print(done.stdout or done.stderr)


[phase] 1 (Development): 30% of the data, 900s for predict()
[data] _sandbox/unicef against data/unicef.json: 5963 respondents, 1529 held out, 18348 cells
[install] baseline/marginal_counts/requirements.txt, network up
[run] sockets disabled in-process, network off

PASS  0.1900  (18.9s)



That last line is the whole of what a submission gets back: `PASS`, the score,
and how long the run took. Nothing else leaves the grader.

The score is **skill**: 0 for a uniform guess, 1 for perfect. Beating the
baseline means using something the crowd shares do not -- the skip logic first
(a gated item's answer is determined whenever its parent is visible), then
whatever the `GIVEN` block tells you about a respondent you have never seen.

To write your own, copy `baseline/marginal_counts/`, edit `predict()`, and point
`--submission` at your copy. Run it against `data/world_bank.json` and
`data/unhcr.json` too: a `predict()` that assumes one instrument's shape fails
on the others.
